<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/RAG_countries_NA_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Example (North + South America): ChromaDB + Gemini

This notebook is an Americas-only variant of the full RAG demo, designed to run faster in live class settings.

Learning goals:
- Understand data ingestion and document construction
- Learn recursive chunking for retrieval quality
- Build a vector index with ChromaDB
- Ground LLM generation using retrieved context

Scope for this version: only countries in North America and South America are indexed.

In [34]:
# Install packages used by the current LangChain ecosystem and provider fallback logic.
# Note: text splitters are now in a dedicated package: langchain-text-splitters
%pip install -q -U google-genai openai chromadb langchain-community langchain-text-splitters pandas requests

## 1) Imports and Environment Setup

This section imports core libraries and handles API key lookup.

In Google Colab, you can store `GEMINI_API_KEY` in Secrets.
Outside Colab, the code falls back to environment variables.

### Functions: provider and API-key helpers

These helpers load credentials and route generation requests.

Lookup order and behavior:
1. Use `GEMINI_API_KEY` when available.
2. If Gemini returns a quota-exhausted message, fall back to `OLLAMA_API_KEY`.
3. For local testing, `.env` can provide both sets of values.

In [35]:
import os
import pandas as pd
import requests
import chromadb
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai



In [51]:
from pathlib import Path

# Load provider keys from a local .env file first for VS Code testing.
env_path = Path.cwd() / ".env"
if env_path.exists():
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

# Try Colab Secrets for both Gemini and Ollama, then fall back to environment variables.
try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None

def _colab_secret(name):
    """Return a Colab secret by name, or None if unavailable."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

def get_gemini_api_key():
    return os.getenv("GEMINI_API_KEY") or _colab_secret("GEMINI_API_KEY")


def get_ollama_api_key():
    """Return the Ollama Cloud API key from env/.env or Colab Secrets."""
    return os.getenv("OLLAMA_API_KEY") or _colab_secret("OLLAMA_API_KEY")


def get_ollama_base_url():
    # Default: Ollama Cloud (https://api.ollama.com/v1).
    # Override with OLLAMA_BASE_URL for a local instance or other endpoint.
    return os.getenv("OLLAMA_BASE_URL", "https://ollama.com/v1")


def get_ollama_model(default_model="deepseek-v3.1:671b-cloud"):
    return os.getenv("OLLAMA_MODEL", default_model)


def has_llm_provider():
    return bool(get_gemini_api_key() or get_ollama_api_key())


def is_gemini_quota_error(exc):
    message = str(exc).lower()
    quota_signals = (
        "quota",
        "resource_exhausted",
        "429",
        "rate limit",
        "exceeded your current quota",
    )
    return any(signal in message for signal in quota_signals)


def generate_text(prompt, system_prompt=None, gemini_model="gemini-2.5-flash", ollama_model=None):
    gemini_api_key = get_gemini_api_key()
    ollama_api_key = get_ollama_api_key()
    ollama_url = get_ollama_base_url()

    if gemini_api_key:
        try:
            client = genai.Client(api_key=gemini_api_key)
            contents = prompt if not system_prompt else f"{system_prompt}\n\n{prompt}"
            response = client.models.generate_content(
                model=gemini_model,
                contents=contents,
            )
            return response.text, "gemini"
        except Exception as exc:
            if not is_gemini_quota_error(exc):
                raise
            if not ollama_api_key:
                raise RuntimeError(
                    "Gemini quota appears exhausted and OLLAMA_API_KEY is not set. "
                    "Add your Ollama Cloud key as a Colab Secret named OLLAMA_API_KEY."
                ) from exc
            print(f"Gemini quota appears exhausted. Falling back to Ollama at {ollama_url} ...")

    if ollama_api_key:
        client = OpenAI(api_key=ollama_api_key, base_url=ollama_url)
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        response = client.chat.completions.create(
            model=ollama_model or get_ollama_model(),
            messages=messages,
        )
        return response.choices[0].message.content, "ollama"

    raise ValueError("Neither GEMINI_API_KEY nor OLLAMA_API_KEY is configured.")


## 2) Build a Minimal RAG Pipeline

Pipeline phases:
1. Data ingestion from GitHub JSON documents
2. Chunking with overlap to preserve context continuity
3. Vector storage and semantic retrieval in ChromaDB
4. Grounded generation with Gemini

### Function: `_collect_text_lines`

This recursive helper converts nested JSON into flat text lines that can be embedded and retrieved.

Why it matters: vector databases work on text, so structured JSON must be turned into readable, searchable content first.

In [37]:
# --- PHASE 1: DATA INGESTION (Pandas + GitHub) ---
def _collect_text_lines(obj, parent_key=""):
    """Recursively flatten nested JSON into readable key-value text lines."""
    lines = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            next_key = f"{parent_key}.{k}" if parent_key else str(k)
            lines.extend(_collect_text_lines(v, next_key))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            next_key = f"{parent_key}[{i}]" if parent_key else f"[{i}]"
            lines.extend(_collect_text_lines(item, next_key))
    else:
        if obj is None:
            return []
        text = str(obj).strip()
        if text:
            lines.append(f"{parent_key}: {text}")
    return lines




### Function: `_get_country_json_paths`

This function discovers country JSON files from the GitHub repository and keeps only North America and South America.

Why it matters: this scoped filter reduces total documents and makes the classroom demo run faster.

In [38]:
def _get_country_json_paths():
    """Discover country JSON files, restricted to North America and South America."""
    tree_url = "https://api.github.com/repos/factbook/factbook.json/git/trees/master?recursive=1"
    r = requests.get(tree_url, timeout=30)
    r.raise_for_status()
    tree = r.json().get("tree", [])

    allowed_prefixes = ("north-america/", "south-america/")
    country_paths = []
    for item in tree:
        path = item.get("path", "")
        if item.get("type") == "blob" and path.endswith(".json") and "/" in path:
            if path.lower().startswith(allowed_prefixes):
                country_paths.append(path)

    # Deduplicate and sort for stable classroom demos.
    return sorted(set(country_paths))




### Function: `get_world_data`

This function fetches each country record, flattens it into text, and packages it with metadata.

Why it matters: it creates the retrieval-ready documents that the rest of the RAG pipeline depends on.

In [39]:
def get_world_data():
    """Fetch all country records and convert each into one retrieval-ready full-text document."""
    print("Step 1: Discovering country files from GitHub...")
    base_url = "https://raw.githubusercontent.com/factbook/factbook.json/master"
    country_paths = _get_country_json_paths()
    print(f"Discovered {len(country_paths)} country files.")

    docs = []
    for path in country_paths:
        try:
            r = requests.get(f"{base_url}/{path}", timeout=20)
            r.raise_for_status()
            payload = r.json()
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        lines = _collect_text_lines(payload)
        if not lines:
            continue

        country_code = path.split("/")[-1].replace(".json", "").upper()
        combined_text = (
            f"COUNTRY_CODE: {country_code}\n"
            f"SOURCE_PATH: {path}\n\n"
            + "\n".join(lines)
        )

        docs.append(
            {
                "content": combined_text,
                "metadata": {
                    "country": country_code,
                    "path": path,
                },
            }
        )

    print(f"Loaded {len(docs)} country documents.")
    return docs




### Function: `chunk_documents`

This function splits long country documents into overlapping chunks before indexing.

Why it matters: smaller chunks improve retrieval precision, while overlap reduces the chance of losing context at chunk boundaries.

In [40]:
## --- PHASE 2: RECURSIVE CHUNKING ---
def chunk_documents(docs, chunk_size=800, chunk_overlap=100):
    """Chunk long records so semantic retrieval can match smaller relevant spans."""
    print("Step 2: Chunking with overlap...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks, metadatas, ids = [], [], []
    for doc in docs:
        split_texts = splitter.split_text(doc["content"])
        for i, text in enumerate(split_texts):
            chunks.append(text)
            metadatas.append(doc["metadata"])
            ids.append(f"{doc['metadata']['country']}_{i}")
    return chunks, metadatas, ids


# Small inspection check for students before moving on.
docs_preview = get_world_data()
print(f"Loaded {len(docs_preview)} source documents.")
print("Example metadata:", docs_preview[0]["metadata"] if docs_preview else "No documents loaded")

Step 1: Discovering country files from GitHub...
Discovered 21 country files.
Loaded 21 country documents.
Loaded 21 source documents.
Example metadata: {'country': 'BD', 'path': 'north-america/bd.json'}


### Checkpoint 1: Data and Chunking Reflection

1. Why do we transform structured JSON fields into one text document per country before retrieval?
2. How might changing `chunk_size` from 800 to 300 affect precision and recall?
3. What is the purpose of overlap, and what failure mode does it reduce?

### 2A) Build and Populate the Vector Store

In this phase, we transform chunks into retrievable vector records inside ChromaDB.

Teaching note: for reproducible classroom runs, we clear the collection before re-inserting documents.

### Function: `build_vector_store`

This function creates the Chroma collection, clears older demo records, and writes new chunk batches into the vector store.

Why it matters: it is the bridge between text preprocessing and semantic retrieval.

In [41]:
def build_vector_store(docs, db_path="./factbook_db", collection_name="world_facts", batch_size=5000):
    """Create (or reset) a Chroma collection and upsert chunked records in batches."""
    chunks, metadatas, ids = chunk_documents(docs)

    chroma_client = chromadb.PersistentClient(path=db_path)
    collection = chroma_client.get_or_create_collection(name=collection_name)

    # Clear older records so each class run starts cleanly.
    existing = collection.get(include=[])
    if existing.get("ids"):
        collection.delete(ids=existing["ids"])

    print("Step 3: Storing vectors in ChromaDB...")
    for start in range(0, len(ids), batch_size):
        stop = start + batch_size
        collection.upsert(
            documents=chunks[start:stop],
            metadatas=metadatas[start:stop],
            ids=ids[start:stop],
        )

    print(f"Stored {len(ids)} chunks in collection '{collection_name}'.")
    return collection

collection = build_vector_store(docs_preview)

Step 2: Chunking with overlap...
Step 3: Storing vectors in ChromaDB...
Stored 1236 chunks in collection 'world_facts'.


### Checkpoint 2: Vector Store Reflection

1. Why do we clear old records before upserting during a classroom demo?
2. What reproducibility issue appears if we repeatedly insert without cleanup?
3. If this were production, when would you keep historical vectors instead?

### 2B) Retrieve Context and Generate a Grounded Answer

This phase demonstrates the core RAG idea: retrieve relevant evidence first, then condition the LLM on that evidence.

Prompting rule used here: answer only from retrieved context; otherwise say you do not know.

### Functions: `answer_with_rag` and `answer_without_rag`

These two functions create the core teaching comparison in the notebook.

The prompting logic stays almost the same.
The main change is the model-call layer: Gemini is tried first, and Ollama is used automatically if Gemini quota is exhausted.

In [47]:
def answer_with_rag(
    question,
    collection,
    n_results=4,
    model_name="gemini-2.5-flash",
    ollama_model=None,
):
    """Retrieve nearest chunks, then answer using Gemini with Ollama fallback on quota exhaustion."""
    print(f"Step 4: Retrieving context for: '{question}'")
    results = collection.query(query_texts=[question], n_results=n_results)
    context = "\n\n".join(results["documents"][0])

    print("Step 5: Generating grounded answer...")
    system_prompt = "You are a careful analyst. Use ONLY the provided context to answer. If the answer is not in the context, say you don't know."
    user_prompt = f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"
    answer_text, provider = generate_text(
        prompt=user_prompt,
        system_prompt=system_prompt,
        gemini_model=model_name,
        ollama_model=ollama_model,
    )
    print(f"Provider used: {provider}")
    return answer_text, context


def answer_without_rag(question, model_name="gemini-2.5-flash", ollama_model=None):
    """Direct LLM answer without retrieval, with Ollama fallback if Gemini quota is exhausted."""
    answer_text, provider = generate_text(
        prompt=question,
        gemini_model=model_name,
        ollama_model=ollama_model,
    )
    print(f"Provider used: {provider}")
    return answer_text

### Checkpoint 3: Retrieval and Prompting Reflection

1. Why do we retrieve first and generate second in a RAG system?
2. What is the effect of changing `n_results` from 2 to 5?
3. How does the prompt reduce hallucinations, and where can it still fail?

### 2C) Run an End-to-End Query

You can now test questions and inspect both the answer and retrieved evidence.

Tip: print retrieved context in class to discuss why the model answered the way it did.

### Function: `preview`

This small helper trims long model outputs and retrieved context so the classroom output stays readable.

Why it matters: it keeps the demo focused on interpretation instead of overwhelming students with raw text.

In [52]:
comparison_questions = [
    {
        "type": "Hallucination test (out-of-scope)",
        "question": "What is the current prime minister of Atlantis, and what is Atlantis GDP growth?",
    },
    {
        "type": "Local info extraction",
        "question": "According to the Factbook data, summarize Mexico's economy overview in 3 bullet points.",
    },
    {
        "type": "Local info extraction",
        "question": "Using the Factbook country records, compare inflation-related details for Brazil and Canada.",
    },
    {
        "type": "Local + public knowledge blend",
        "question": "Using Factbook context and your general knowledge, explain how Mexico's economic profile may influence its role in global supply chains today.",
    },
]


def preview(text, max_len=900):
    text = text or ""
    return text[:max_len] + ("..." if len(text) > max_len else "")


if "collection" not in globals():
    print("Error: Build the Chroma vector store in Cell 19 before running this comparison.")
elif not has_llm_provider():
    print("Error: Set GEMINI_API_KEY or OLLAMA_API_KEY before running the model comparison.")
else:
    try:
        for i, item in enumerate(comparison_questions, start=1):
            q_type = item["type"]
            q = item["question"]
            print("\n" + "=" * 90)
            print(f"Q{i}. {q_type}")
            print(f"QUESTION: {q}\n")

            direct_answer = answer_without_rag(q)
            print("[WITHOUT RAG]")
            print(preview(direct_answer))

            rag_answer, rag_context = answer_with_rag(q, collection, n_results=4)
            print("\n[WITH RAG]")
            print(preview(rag_answer))

            print("\n[RETRIEVED CONTEXT PREVIEW]")
            print(preview(rag_context, max_len=1200))
    except Exception as e:
        print(f"Error: Check GEMINI_API_KEY / OLLAMA_API_KEY configuration. Details: {e}")


Q1. Hallucination test (out-of-scope)
QUESTION: What is the current prime minister of Atlantis, and what is Atlantis GDP growth?

Gemini quota appears exhausted. Falling back to Ollama at https://ollama.com/v1 ...
Provider used: ollama
[WITHOUT RAG]
Your question references "Atlantis," which is traditionally understood as a fictional island—first mentioned in Plato’s dialogues—and not a real country with a government or economic statistics.  

If this is related to a **fictional work** (like a book, game, or film) that includes a modern Atlantis with political and economic details, please let me know which specific version or universe you're referring to, and I’d be happy to help explore that setting.  

Otherwise, in real-world terms, Atlantis does not exist, so there is no current prime minister or measurable GDP growth.
Step 4: Retrieving context for: 'What is the current prime minister of Atlantis, and what is Atlantis GDP growth?'
Step 5: Generating grounded answer...
Gemini quot

### Checkpoint 4: Evaluation Reflection

1. Compare the final answer against the retrieved context. Which statements are directly supported?
2. What additional evidence would improve confidence in the answer?
3. Design one out-of-scope query and explain how the system should respond.

## 3) Teaching Notes and Suggested Exercises

Exercises for class discussion:
- Change `chunk_size` and `chunk_overlap`; evaluate answer quality
- Increase `n_results` and observe grounding changes
- Add more countries and test comparative questions
- Intentionally ask out-of-scope questions to inspect hallucination control